# Persiapan Batch-02

Memeriksa tiga dataset terbaru setelah penggantian Mon dan Alif, membekukan input evaluasi, serta memverifikasi aliran sensor pendamping. Notebook ini tidak melatih model.


## Lokasi proyek

Notebook membaca data dan menyimpan hasil melalui Google Drive. Path utama adalah `/content/drive/MyDrive/Penelitian-Adaptive-IDE`. Tidak ada upload CSV melalui notebook.


In [ ]:
from pathlib import Path
import os
import sys
from google.colab import drive

drive.mount('/content/drive')
IN_COLAB = True
PROJECT_ROOT = Path('/content/drive/MyDrive/Penelitian-Adaptive-IDE')
BATCH_DIR = PROJECT_ROOT / 'models' / 'batch-02'
if not (BATCH_DIR / 'shared' / 'batch02').is_dir():
    raise FileNotFoundError('Salin folder batch-02, termasuk shared, requirements.txt, dan sensor_streams ke Drive.')
sys.path.insert(0, str(BATCH_DIR / 'shared'))


## Environment

Dependency inti diperiksa dan dipasang bila diperlukan. Jika diminta restart runtime setelah instalasi, restart sekali lalu jalankan dari awal. TabPFN tetap opsional dan tidak mengunduh checkpoint otomatis.


In [ ]:
from batch02.runtime import ensure_environment

# Gunakan runtime Colab 2026.10 (Python 3.13) atau runtime Python >=3.12.
ENABLE_TABPFN = False
INSTALL_DEPENDENCIES = True
OPTIONAL_GROUPS = ()
environment_status = ensure_environment(BATCH_DIR, install=INSTALL_DEPENDENCIES, optional=OPTIONAL_GROUPS)
print(environment_status)
if environment_status['status'] not in ('ready', 'installed'):
    raise RuntimeError('Dependency belum sesuai. Periksa hasil cell Environment.')


## Konfigurasi

Ketiga panjang window diproses oleh notebook yang sama. `RESUME_DIR = None` membuat run baru. Isi path run hanya untuk melanjutkan run Colab dengan data, kode, konfigurasi, dan environment yang sama.


In [ ]:
from batch02.contracts import Config

EXPERIMENT = 'exp00'
DATASET_ROOT = PROJECT_ROOT / 'Dataset' / 'v2_session_phase'
RESUME_DIR = None
CPU_THREADS = max(1, min(4, os.cpu_count() or 1))
config = Config(PROJECT_ROOT, dataset_root=DATASET_ROOT, threads=CPU_THREADS)
print('Dataset:', config.dataset)
print('Output:', config.batch / 'outputs' / EXPERIMENT)
print('CPU threads:', CPU_THREADS)


## Pemeriksaan Dataset

Membaca ketiga CSV dan allowlist, memeriksa label/interval/fitur, lalu menampilkan proporsi aktual serta gate yang belum disetujui. Dataset sekarang boleh diperiksa; pemeriksaan ini tidak menyatakan data tambahan sudah masuk.


In [ ]:
from batch02.runner import readiness

readiness_report = readiness(config)
display(readiness_report['dataset'])
display(readiness_report['release'])
display(readiness_report['capabilities'])


## Registry Identitas Lintas Sesi

Tinjau person_id yang berasal dari metadata. Jika seseorang mempunyai beberapa sesi, gunakan person_id yang sama melalui PERSON_ALIASES. Konfirmasi registry hanya setelah identitasnya diperiksa; kolom ini tidak menjadi fitur model.


In [ ]:
from batch02.runner import registry_draft

session_registry = registry_draft(config)
PERSON_ALIASES = {}
CONFIRM_IDENTITY_REGISTRY = True
ADDITIONAL_SESSION_IDS = [
    '20261116_161850_P01_coding_task',
    '20261116_154424_P01_coding_task',
    '20261117_143302_P01_coding_task',
    '20261117_153818_P01_coding_task',
]
for row in session_registry['sessions']:
    row['person_id'] = PERSON_ALIASES.get(row['session_id'], row['person_id'])
    row['confirmed'] = CONFIRM_IDENTITY_REGISTRY
    row['collection_wave'] = 'additional_collection' if row['session_id'] in ADDITIONAL_SESSION_IDS else 'original_collection'
display(session_registry)


## Pembekuan input evaluasi

Salinan dataset, registry, dan split disimpan di `inputs/<release_id>`. Persetujuan berlaku untuk hash dataset terbaru yang tercantum di bawah. Pengulangan pada input yang sama memuat release yang sudah dibuat.


In [ ]:
from batch02.contracts import freeze_release, inspect_dataset, load_release
from batch02.artifacts import read_json

APPROVE_RELEASE = True
PROPORTIONS_APPROVED = True
APPROVED_BY = 'Tim penelitian; dataset setelah Mon dan Alif, 17 November 2026'
APPROVED_MANIFEST_SHA256 = '5bc566a20c7aa67388da4cf19a0587791af94ba650c33eac785a6ea9989e4d52'

if APPROVE_RELEASE:
    inspected = inspect_dataset(config)
    if inspected['files']['manifest.json'] != APPROVED_MANIFEST_SHA256:
        raise ValueError('Dataset berbeda dari paket notebook ini. Gunakan dataset dan sensor_streams yang sama; tinjau ulang jika dataset diperbarui.')
    release_path = config.batch / 'registry' / 'dataset_release.json'
    existing = read_json(release_path) if release_path.exists() else {}
    if existing.get('files', {}).get('manifest.json') == APPROVED_MANIFEST_SHA256:
        dataset_release = load_release(config)[0]
    else:
        dataset_release = freeze_release(
            config, session_registry, ADDITIONAL_SESSION_IDS, APPROVED_BY,
            proportions_approved=PROPORTIONS_APPROVED,
            evaluation_scope='colab_latest_dataset', collection_status='ongoing',
            expected_manifest_sha256=APPROVED_MANIFEST_SHA256,
        )
    display(dataset_release)
else:
    raise RuntimeError('Release belum disahkan; training tetap diblokir.')


## Aliran sensor dan beban eksperimen

Paket `sensor_streams` dibaca dari Drive dan dicocokkan dengan dataset. Aliran memuat seluruh grid sebelum pemilihan target, sehingga history tidak dibentuk dari CSV filtered. Frame eye bertimestamp ambigu tidak dipakai sebagai fitur konteks. Notebook juga menghitung jumlah pekerjaan tanpa fitting model.


In [ ]:
from batch02.stream_adapter import attach_portable_streams
from batch02.contracts import load_release
from batch02.artifacts import write_json
from batch02.runtime import capabilities
from batch02.splits import make_manifests, paired_endpoints, restrict_paired
from batch02.workload import estimate_work

stream_verification = attach_portable_streams(config, BATCH_DIR / 'sensor_streams')
display(stream_verification)
release, frames, feature_allowlist, split_manifest = load_release(config)
run_dir = config.batch / 'outputs' / 'exp00' / release['release_id']
run_dir.mkdir(parents=True, exist_ok=True)
paired = paired_endpoints(frames)
paired_manifests = make_manifests({s: restrict_paired(f, paired) for s, f in frames.items()}, config.seed)
workload = {f'exp{i:02d}': estimate_work(f'exp{i:02d}', split_manifest,
            paired_manifests=paired_manifests, capabilities=capabilities(config)) for i in range(1, 15)}
write_json(run_dir / 'workload.json', workload)
write_json(run_dir / 'release.json', release)
write_json(run_dir / 'stream_verification.json', stream_verification)
write_json(run_dir / 'status.json', {'status': 'prepared', 'training_performed': False,
           'release_id': release['release_id']})
print('Persiapan selesai:', run_dir)
print('Lanjutkan exp01 sampai exp14, satu notebook pada satu waktu.')
